In [0]:
# importar librerias
from pyspark.sql import functions as F
from delta.tables import DeltaTable

FACT_TRANSACTIONS = "banco.banking_analytics.fact_transactions"

In [0]:
# Analizar el plan de ejecución
df_perf_baseline = (
    spark.table(FACT_TRANSACTIONS)
    .filter(F.col("status") == "COMPLETED")
    .groupBy("currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount")
    )
)

df_perf_baseline.explain("formatted")

In [0]:
display(df_perf_baseline)

In [0]:
# Medir el tiempo de ejecución
import time

start_time = time.perf_counter()

results = df_perf_baseline.collect()

end_time = time.perf_counter()

execution_time = end_time - start_time

print(f"Tiempo de ejecución: {execution_time:.4f} segundos")

In [0]:
# Analizar estadisticas de ejecución
df_perf_baseline.explain("cost")

In [0]:
# Verificar estadisticas de la tabla
spark.sql("""
    DESCRIBE DETAIL banco.banking_analytics.fact_transactions
""").display()

In [0]:
%sql
USE CATALOG banco;
SHOW TABLE EXTENDED IN banco.banking_analytics
LIKE 'fact_transactions';


In [0]:
df_file_sizes = spark.sql(f"""
    DESCRIBE DETAIL {FACT_TRANSACTIONS}
""").select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns",
    "clusteringColumns"
)

display(df_file_sizes)

In [0]:
display(
    spark.table(FACT_TRANSACTIONS)
    .select("transaction_id")
    .limit(10)
)

In [0]:
# Simular Small File Problems
df_small_files = (
    spark.table(FACT_TRANSACTIONS)
    .repartition(20)
)

print(f"Particiones del DataFrame: {df_small_files.select(F.spark_partition_id()).distinct().count()}")
print(f"Registros: {df_small_files.count()}")

In [0]:
# Generar archivos pequeños
SMALL_FILES_TABLE = "banco.banking_analytics.fact_transactions_small_files"

(
    df_small_files.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SMALL_FILES_TABLE)
)

print("Tabla experimental creada correctamente.")

In [0]:
# Inspeccionar los pequeños archivos generados
df_small_detail = spark.sql(f"""
    DESCRIBE DETAIL {SMALL_FILES_TABLE}
""")

display(
    df_small_detail.select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns"
    )
)

In [0]:
# Aplicando OPTIMIZE
spark.sql(f"""
    OPTIMIZE {SMALL_FILES_TABLE}
""").display()

In [0]:
display(
    spark.sql(f"DESCRIBE DETAIL {SMALL_FILES_TABLE}")
    .select("numFiles", "sizeInBytes")
)

In [0]:
display(
    DeltaTable.forName(spark, SMALL_FILES_TABLE)
    .history(5)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .orderBy(F.col("version").desc())
)

In [0]:
df_after_optimize = spark.table(SMALL_FILES_TABLE)

print(f"Total de registros: {df_after_optimize.count()}")

print(
    "Transacciones únicas:",
    df_after_optimize
    .select("transaction_id")
    .distinct()
    .count()
)

In [0]:
SMALL_FILES_TABLE = "banco.banking_analytics.fact_transactions_small_files"

spark.sql(f"""
DESCRIBE DETAIL {SMALL_FILES_TABLE}
""").display()

In [0]:
spark.sql(f"""
OPTIMIZE {SMALL_FILES_TABLE}
ZORDER BY (customer_key, transaction_date)
""").display()

In [0]:
# Comparar Z-Order con Liquid Clustering
ZORDER_TABLE = "banco.banking_analytics.fact_transactions_zorder"

LIQUID_TABLE = "banco.banking_analytics.fact_transactions_liquid"

In [0]:
# Tabla para aplicar Z-order
(
    spark.table(FACT_TRANSACTIONS)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(ZORDER_TABLE)
)

In [0]:
# Tabla para aplicar liquid clustering
spark.sql(f"""
CREATE OR REPLACE TABLE {LIQUID_TABLE}
CLUSTER BY (customer_key, transaction_date)
AS
SELECT *
FROM {FACT_TRANSACTIONS}
""")

In [0]:
# Validar integridad de datos
tables_to_validate = [
    FACT_TRANSACTIONS,
    ZORDER_TABLE,
    LIQUID_TABLE
]

for table_name in tables_to_validate:
    df = spark.table(table_name)

    total_rows = df.count()
    unique_transactions = (
        df.select("transaction_id")
        .distinct()
        .count()
    )

    print(f"\nTabla: {table_name}")
    print(f"Total registros: {total_rows}")
    print(f"Transacciones únicas: {unique_transactions}")

In [0]:
# Inspeccionar ambas tablas
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE DETAIL {table_name}
    """).select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "clusteringColumns",
        "clusterByAuto"
    ).display()

In [0]:
# Aplicando Zorder
spark.sql(f"""
OPTIMIZE {ZORDER_TABLE}
ZORDER BY (customer_key, transaction_date)
""").display()

In [0]:
# Ejecutar la tabla liquid
spark.sql(f"""
OPTIMIZE {LIQUID_TABLE}
""").display()

In [0]:
# Historial Delta
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE HISTORY {table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).display()

In [0]:
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE DETAIL {table_name}
    """).select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "clusteringColumns"
    ).display()